In [2]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

In [3]:
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [4]:
df.drop(columns=['id', 'Unnamed: 32'], inplace =True)
df.head()

,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [5]:
X_train, X_test, y_train, y_test = train_test_split(df.iloc[:,1:], df.iloc[:,0], test_size=0.2)

In [6]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [7]:
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

numpy to tensors

In [8]:
X_train_tensor = torch.from_numpy(X_train.astype(np.float32))
X_test_tensor = torch.from_numpy(X_test.astype(np.float32))
y_train_tensor = torch.from_numpy(y_train.astype(np.float32))
y_test_tensor = torch.from_numpy(y_test.astype(np.float32))

In [14]:
from torch.utils.data import Dataset, DataLoader

class CustomDataset(Dataset):
    def __init__(self, features, labels):
        self.features = features
        self.labels = labels
    
    def __len__(self):
        return self.features.shape[0]
    
    def __getitem__(self, index):
        return self.features[index], self.labels[index]

In [15]:
train_dataset = CustomDataset(X_train_tensor, y_train_tensor)
test_dataset = CustomDataset(X_test_tensor, y_test_tensor)

In [16]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=True)

define model

In [9]:
import torch.nn as nn

class MySingleNN(nn.Module):
    def __init__(self, features):
        super().__init__()
        self.linear = nn.Linear(features, 1)
        self.sigmoid = nn.Sigmoid() 
    
    def forward(self, features):
        linear_out = self.linear(features)
        out = self.sigmoid(linear_out)
        return out

define parameters

In [10]:
learning_rate = 0.1
epochs = 25

training pipeline

In [17]:
import torch.optim as optim

# create model
model = MySingleNN(X_train_tensor.shape[1])

# define optimizer
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

# define loss function
loss_function = nn.BCELoss()

In [27]:
# define loop
for epoch in range(epochs):
        for batch_features, batch_labels in train_loader:
        #1. forward pass
            y_predict = model(batch_features)

        #2. loss calculate
            loss = loss_function(y_predict, batch_labels.view(-1,1))

        #clear gradients
            optimizer.zero_grad()
    
        #3. backward pass, calculate derivatives
            loss.backward()
        
        #4. param update
            optimizer.step()
            print(f"Epoch: {epoch}, and the loss with it: {loss}")   

Epoch: 0, and the loss with it: 0.06672768294811249
Epoch: 0, and the loss with it: 0.015202955342829227
Epoch: 0, and the loss with it: 0.20177215337753296
Epoch: 0, and the loss with it: 0.02904520370066166
Epoch: 0, and the loss with it: 0.032273486256599426
Epoch: 0, and the loss with it: 0.03050195425748825
Epoch: 0, and the loss with it: 0.09246300160884857
Epoch: 0, and the loss with it: 0.045366864651441574
Epoch: 0, and the loss with it: 0.10571122914552689
Epoch: 0, and the loss with it: 0.015209725126624107
Epoch: 0, and the loss with it: 0.03968985751271248
Epoch: 0, and the loss with it: 0.01786429062485695
Epoch: 0, and the loss with it: 0.013519306667149067
Epoch: 0, and the loss with it: 0.03410504385828972
Epoch: 0, and the loss with it: 0.0035755776334553957
Epoch: 1, and the loss with it: 0.0052512530237436295
Epoch: 1, and the loss with it: 0.011943749152123928
Epoch: 1, and the loss with it: 0.05751584842801094
Epoch: 1, and the loss with it: 0.10555454343557358
Ep

evaluation

In [30]:
model.eval() #Set model to evaluation mode, not training anymore
accuracy_list = []

with torch.no_grad():
    for batch_features, batch_labels in test_loader:
        y_predict = model(batch_features)
        y_predict = (y_predict > 0.8).float()

        batch_accuracy = (y_predict.view(-1) == batch_labels).float().mean().item()
        accuracy_list.append(batch_accuracy)

overall_accuracy =  sum(accuracy_list)/len(accuracy_list)
print(f'Accuracy: {overall_accuracy}')

Accuracy: 0.9704861044883728
